# 02 — Improving crack recall

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hasanbahcecii/surface-anomaly-detector/blob/main/notebooks/02_improve_recall_colab.ipynb)

The baseline (notebook 01) misses ~26% of cracked patches, and a lower threshold only buys recall with ~9× more
false alarms. This run changes two things against the baseline:

| | Baseline | v2 |
|---|---|---|
| Input size | 224 (SDNET patches downscaled) | **256** (native, keeps hairline cracks) |
| Cracked share in train | 15% | **~26%** (each cracked train image twice) |

Everything else (model, epochs, seed, val/test split) is identical, so the difference is attributable to these two changes.

**Runtime → T4 GPU**, then Run all. Needs the `KAGGLE_USERNAME` / `KAGGLE_KEY` secrets and the baseline run on Drive.
Training takes ~2.5 h; checkpoints go to Drive, so a disconnect can be resumed (section 3).

## 1. Setup (safe to re-run; skips what this session already has)

In [ ]:
import os
REPO_URL = "https://github.com/hasanbahcecii/surface-anomaly-detector.git"
REPO = "/content/surface-anomaly-detector"
DRIVE_OUT = "/content/drive/MyDrive/surface-anomaly-detector"
BASE_DIR = f"{DRIVE_OUT}/runs/baseline"
RUN_NAME = "v2_256_os2"
RUN_DIR = f"{DRIVE_OUT}/runs/{RUN_NAME}"
DATA = "data/processed_os2"
IMGSZ = 256

if not os.path.exists(REPO):
    !git clone -q {REPO_URL} {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!pip install -q -r requirements.txt

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

if not os.path.exists(f"{DATA}/test"):
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    if not os.path.exists("data/raw"):
        !kaggle datasets download -d aniruddhsharma/structural-defects-network-concrete-crack-images -p data/ --unzip -q
        !find data -maxdepth 1 -mindepth 1 -type d ! -name raw ! -name 'processed*' -exec mv {} data/raw \;
    !python scripts/prepare_data.py --src data/raw --out {DATA} --seed 42 --oversample-cracked 2
!nvidia-smi --query-gpu=name --format=csv,noheader

## 2. Look at the baseline's mistakes first (~2 min)

If most missed cracks are clearly visible, the model is the problem and this run should help.
If they look uncracked (label noise), no model change will fix them — worth knowing before spending 2.5 h.

In [ ]:
from IPython.display import Image, display
!python scripts/show_errors.py --weights {BASE_DIR}/weights/best.pt --data {DATA} --out-dir {BASE_DIR}/errors --imgsz 224
display(Image(f"{BASE_DIR}/errors/missed_cracks.png"))
display(Image(f"{BASE_DIR}/errors/false_alarms.png"))

## 3. Train v2

After a disconnect: re-run section 1, then set `RESUME = True` here and run this cell again.

In [ ]:
from ultralytics import YOLO

RESUME = False
if RESUME:
    model = YOLO(f"{RUN_DIR}/weights/last.pt")
    model.train(resume=True)
else:
    model = YOLO("yolov8n-cls.pt")
    model.train(
        data=DATA,
        epochs=30,
        imgsz=IMGSZ,
        batch=128,
        patience=8,
        seed=42,
        project=f"{DRIVE_OUT}/runs",
        name=RUN_NAME,
        exist_ok=True,
    )
print("saved to", model.trainer.save_dir)

## 4. Test metrics, threshold tuning and the v2 mistakes

In [ ]:
!python scripts/evaluate.py --weights {RUN_DIR}/weights/best.pt --data {DATA} --split test --imgsz {IMGSZ} --out {RUN_DIR}/test_metrics.json
!python scripts/tune_threshold.py --weights {RUN_DIR}/weights/best.pt --data {DATA} --out-dir {RUN_DIR} --imgsz {IMGSZ} --target-recall 0.90
!python scripts/show_errors.py --weights {RUN_DIR}/weights/best.pt --data {DATA} --out-dir {RUN_DIR}/errors --imgsz {IMGSZ}
display(Image(f"{RUN_DIR}/threshold_curve.png"))

## 5. Baseline vs v2 (same test split)

In [ ]:
import json

def summary(run_dir):
    m = json.load(open(f"{run_dir}/test_metrics.json"))
    cm = m["confusion_matrix"]  # rows/cols ordered by class index: cracked, uncracked
    t = json.load(open(f"{run_dir}/threshold.json"))
    hi = next(c for c in t["candidates"] if c["name"].startswith("recall"))["test"]
    return {
        "P": m["cracked"]["precision"], "R": m["cracked"]["recall"], "F1": m["cracked"]["f1-score"],
        "acc": m["accuracy"], "missed": cm[0][1], "false+": cm[1][0],
        "hiR_thr": hi["threshold"], "hiR_R": hi["recall"], "hiR_P": hi["precision"], "hiR_false+": hi["fp"],
    }

rows = {"baseline": summary(BASE_DIR), RUN_NAME: summary(RUN_DIR)}
keys = list(rows["baseline"])
print(f"{'':<12}" + "".join(f"{k:>11}" for k in keys))
for name, r in rows.items():
    print(f"{name:<12}" + "".join(f"{r[k]:>11.3f}" if isinstance(r[k], float) else f"{r[k]:>11}" for k in keys))